# Group Assignment – Part A Q5: Alternative Tokenisation

**Student:** Chew Aik Yang  
**Alternative approach:** scikit-learn `CountVectorizer` analyser

This notebook addresses all three Q5 requirements:

1. implement an alternative tokenisation approach (**3 marks**);
2. compare it with the group's Q1 approaches (**2 marks**); and
3. explain why it is better, worse, or simply different (**5 marks**).

The same `Data_1.txt` corpus used by the group in Q1 is embedded below so that every method processes identical text and the notebook remains reproducible on another computer.

## 0. Environment Setup

`CountVectorizer` is a common scikit-learn text-processing tool. Its analyser performs lowercasing and word-token extraction before the vectoriser builds a vocabulary and numerical document–term matrix. NLTK is used only to reproduce the group's Q1 comparison method.

In [1]:
import re

import nltk
import pandas as pd
import sklearn
from IPython.display import display
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer


def ensure_nltk_resource(resource_path, package_name):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)


ensure_nltk_resource('tokenizers/punkt', 'punkt')
ensure_nltk_resource('tokenizers/punkt_tab', 'punkt_tab')

print(f"Environment ready — scikit-learn {sklearn.__version__}, NLTK {nltk.__version__}, pandas {pd.__version__}")

Environment ready — scikit-learn 1.8.0, NLTK 3.9.4, pandas 3.0.3


## 1. Text Data

The corpus below is copied exactly from the group's Q1/Q2 notebook. Embedding it removes the machine-specific absolute file paths found in some individual notebooks while preserving the assignment data.

In [2]:
corpus_text = 'Classification is the task of choosing the correct class label for a given input. In basic\nclassification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.'

print("Original Data_1 corpus:\n")
print(corpus_text)

Original Data_1 corpus:

Classification is the task of choosing the correct class label for a given input. In basic
classification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.


## 2. Q5.1 – Alternative Implementation Using CountVectorizer (3 marks)

The vectoriser's `build_analyzer()` method exposes the ordered tokens produced during feature extraction. With the default settings, the analyser:

- converts text to lowercase;
- selects alphanumeric word tokens containing at least two characters;
- removes punctuation as delimiters; and
- splits a hyphenated expression such as `open-class` into `open` and `class`.

After displaying the tokens, `fit_transform()` builds a one-row numerical document–term matrix for this corpus.

In [3]:
vectorizer = CountVectorizer()
analyzer = vectorizer.build_analyzer()

countvectorizer_tokens = analyzer(corpus_text)
document_term_matrix = vectorizer.fit_transform([corpus_text])
feature_names = vectorizer.get_feature_names_out().tolist()

print(f"CountVectorizer token count: {len(countvectorizer_tokens)}")
print(f"Unique vocabulary features: {len(feature_names)}")
print(f"Document–term matrix shape: {document_term_matrix.shape}\n")
print("Complete ordered token output:\n")
print(countvectorizer_tokens)

CountVectorizer token count: 80
Unique vocabulary features: 45
Document–term matrix shape: (1, 45)

Complete ordered token output:

['classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'given', 'input', 'in', 'basic', 'classification', 'tasks', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', 'the', 'basic', 'classification', 'task', 'has', 'number', 'of', 'interesting', 'variants', 'for', 'example', 'in', 'multiclass', 'classification', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', 'in', 'open', 'class', 'classification', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', 'and', 'in', 'sequence', 'classification', 'list', 'of', 'inputs', 'are', 'jointly', 'classified']


In [4]:
feature_frequency = pd.DataFrame({
    'Feature': feature_names,
    'Frequency': document_term_matrix.toarray()[0]
}).sort_values(['Frequency', 'Feature'], ascending=[False, True]).reset_index(drop=True)

print("Vocabulary and frequency evidence:")
display(feature_frequency)

Vocabulary and frequency evidence:


,Feature,Frequency
0,in,7
1,classification,6
2,of,5
3,the,5
4,is,4
5,labels,3
6,advance,2
7,and,2
8,basic,2
9,class,2


### Interpretation of the alternative output

The analyser returns **80 ordered tokens**, while the fitted vocabulary contains **45 unique features**. The difference occurs because repeated tokens, such as `classification`, appear several times in the ordered token list but only once as a vocabulary feature. The matrix shape `(1, 45)` confirms that one document has been represented using 45 numerical feature columns.

## 3. Q5.2 – Comparison with the Group's Q1 Approaches (2 marks)

Q1 used Python `split()`, a regular-expression pattern, and NLTK `word_tokenize()`. The following cells apply all four methods to the same corpus. This makes the comparison evidence-based rather than relying only on general definitions.

In [5]:
split_tokens = corpus_text.split()
regex_tokens = re.findall(r'\b\w+\b', corpus_text)
nltk_tokens = word_tokenize(corpus_text)

comparison_table = pd.DataFrame([
    {
        'Method': 'Python split()',
        'Token count': len(split_tokens),
        'Lowercases automatically': 'No',
        'Punctuation treatment': 'Retained when attached to words',
        'Hyphen treatment': 'Preserves open-class',
        'Main use': 'Quick whitespace separation',
    },
    {
        'Method': 'Regular Expression',
        'Token count': len(regex_tokens),
        'Lowercases automatically': 'No',
        'Punctuation treatment': 'Removed by the selected pattern',
        'Hyphen treatment': 'Splits into open and class',
        'Main use': 'Rule-based word extraction',
    },
    {
        'Method': 'NLTK word_tokenize()',
        'Token count': len(nltk_tokens),
        'Lowercases automatically': 'No',
        'Punctuation treatment': 'Produces separate punctuation tokens',
        'Hyphen treatment': 'Preserves open-class',
        'Main use': 'General linguistic processing',
    },
    {
        'Method': 'CountVectorizer analyser',
        'Token count': len(countvectorizer_tokens),
        'Lowercases automatically': 'Yes',
        'Punctuation treatment': 'Removed as delimiters',
        'Hyphen treatment': 'Splits into open and class',
        'Main use': 'Machine-learning feature preparation',
    },
])

display(comparison_table)

,Method,Token count,Lowercases automatically,Punctuation treatment,Hyphen treatment,Main use
0,Python split(),82,No,Retained when attached to words,Preserves open-class,Quick whitespace separation
1,Regular Expression,83,No,Removed by the selected pattern,Splits into open and class,Rule-based word extraction
2,NLTK word_tokenize(),94,No,Produces separate punctuation tokens,Preserves open-class,General linguistic processing
3,CountVectorizer analyser,80,Yes,Removed as delimiters,Splits into open and class,Machine-learning feature preparation


In [6]:
focused_examples = pd.DataFrame([
    {
        'Example': 'First token / case',
        'split()': split_tokens[0],
        'Regex': regex_tokens[0],
        'NLTK': nltk_tokens[0],
        'CountVectorizer': countvectorizer_tokens[0],
    },
    {
        'Example': 'input followed by a full stop',
        'split()': [t for t in split_tokens if t.startswith('input')][:1],
        'Regex': [t for t in regex_tokens if t == 'input'][:1],
        'NLTK': [t for t in nltk_tokens if t in {'input', '.'}][:2],
        'CountVectorizer': [t for t in countvectorizer_tokens if t == 'input'][:1],
    },
    {
        'Example': 'labels followed by a semicolon',
        'split()': [t for t in split_tokens if t.startswith('labels;')][:1],
        'Regex': [t for t in regex_tokens if t == 'labels'][:1],
        'NLTK': [t for t in nltk_tokens if t in {'labels', ';'}][:2],
        'CountVectorizer': [t for t in countvectorizer_tokens if t == 'labels'][:1],
    },
    {
        'Example': 'open-class',
        'split()': [t for t in split_tokens if 'open-class' in t],
        'Regex': [t for t in regex_tokens if t in {'open', 'class'}][-2:],
        'NLTK': [t for t in nltk_tokens if 'open-class' in t],
        'CountVectorizer': [t for t in countvectorizer_tokens if t in {'open', 'class'}][-2:],
    },
])

print("Focused output comparison:")
display(focused_examples)

Focused output comparison:


,Example,split(),Regex,NLTK,CountVectorizer
0,First token / case,Classification,Classification,Classification,classification
1,input followed by a full stop,[input.],[input],"[input, .]",[input]
2,labels followed by a semicolon,[labels;],[labels],"[labels, labels]",[labels]
3,open-class,[open-class],"[open, class]",[open-class],"[open, class]"


### Comparison and contrast based on the obtained output

- **Token counts differ:** `split()` produces 82 tokens, the regular expression produces 83, NLTK produces 94, and CountVectorizer produces 80. NLTK has the largest count because punctuation such as `.` and `;` becomes separate tokens. CountVectorizer has fewer tokens because punctuation and one-character tokens such as `a` are excluded by its default pattern.
- **Capitalisation differs:** the first three Q1 methods retain `Classification`, whereas CountVectorizer produces `classification`. This automatic normalisation prevents the same word from becoming separate uppercase and lowercase features.
- **Punctuation differs:** `split()` retains forms such as `input.` and `labels;`; NLTK separates the word from its punctuation; the regular expression and CountVectorizer remove punctuation from their word output.
- **Hyphenation differs:** `split()` and NLTK preserve `open-class`, while the chosen regular expression and CountVectorizer return `open` and `class` separately.

Therefore, the methods are not interchangeable. Their outputs reflect different purposes and preprocessing decisions.

## 4. Q5.3 – Why CountVectorizer Is Better, Worse, or Different (5 marks)

### Where it is better

CountVectorizer is better when the next task is conventional machine learning. In this notebook, one fitted object performs lowercasing, token extraction, vocabulary construction and numerical counting. The output immediately becomes a `(1, 45)` sparse document–term matrix, so no separate manual loop is needed to convert the 80 tokens into model features. This is useful for classifiers such as Multinomial Naive Bayes and Logistic Regression.

The displayed first-token comparison also shows a practical advantage: `Classification` becomes `classification`. Treating case variants as the same feature can reduce unnecessary vocabulary duplication when capitalisation is not important to the classification problem.

### Where it is worse

Its defaults discard information. NLTK preserves punctuation as separate tokens, but CountVectorizer removes it. The default CountVectorizer pattern also excludes one-character tokens; therefore, occurrences of `a` shown by the Q1 methods do not appear in its output. These choices may be unsuitable if punctuation, short symbols, capitalisation, or exact token positions carry meaning—for example, in detailed linguistic analysis, authorship analysis, or sentiment involving expressive punctuation.

Hyphenated terms also require care. The evidence table shows that `open-class` becomes two features, `open` and `class`. This can be helpful when both words have independent predictive meaning, but worse when the compound should remain one concept. The `token_pattern` parameter can be customised, although changing it would need to be justified for the dataset.

### Why it is also simply different

The main difference is its intended output. `split()`, regex and NLTK primarily return token sequences. CountVectorizer additionally learns a fixed vocabulary and maps documents to numerical feature vectors. It is therefore more than a display tokenizer: it is an integrated feature-extraction step. NLTK remains more appropriate when detailed linguistic tokens are required, while CountVectorizer is more convenient when clean word counts are the immediate input to a supervised text-classification model.

## 5. Executable Validation

These checks verify the observed CountVectorizer behaviour and the consistency between its ordered tokens, vocabulary and numerical matrix.

In [7]:
assert len(countvectorizer_tokens) == 80
assert all(token == token.lower() for token in countvectorizer_tokens)
assert 'open' in countvectorizer_tokens and 'class' in countvectorizer_tokens
assert 'open-class' not in countvectorizer_tokens
assert 'a' not in countvectorizer_tokens
assert document_term_matrix.shape == (1, len(feature_names))
assert int(document_term_matrix.sum()) == len(countvectorizer_tokens)
assert len(feature_names) == 45

print('All validation checks passed.')

All validation checks passed.


## Conclusion

The alternative implementation is complete and distinct from the other members' BERT, spaCy and Gensim approaches. For this corpus, CountVectorizer produces 80 normalised word tokens and 45 numerical vocabulary features. It is the stronger choice when tokenisation must lead directly into a traditional machine-learning pipeline, but NLTK or a customised tokenizer is preferable when punctuation, one-character terms, capitalisation, or compound-word boundaries must be preserved. The method is therefore better for feature preparation, worse for detailed linguistic fidelity, and different in the type of output it is designed to produce.